# Неделя 1. Домашнее задание по теме «Комбинаторика. Классическая и геометрическая вероятность. Метод Монте-Карло»

## Импорты и настройка

В этой ячейке все необходимые импорты.

**Ничего менять не нужно — просто запусти ячейку.**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
import time
import random
import itertools
from typing import List, Tuple

w = 30

print(f'Импорты загружены.')

Импорты загружены.


# Задача 1 вариант 2 (буква М)

## Задача 1 (3 балла). Комбинаторика и классическая вероятность

Есть кластер из N серверов — каждый со своим ID: SRV_0, SRV_1, ..., SRV_N-1. На кластер поступает 10 запросов в секунду. Каждый запрос отправляется на один из серверов случайным образом.

Система считается перегруженной, если:
- на три сервера пришло 3 запроса и больше;

ИЛИ

- на два сервера пришло 4 запроса и больше.

**Задание:**

**1.1** (0,75 балла). При N = 30 аналитически найди точную вероятность того, что система стабильна.

**1.2** (0,75 балла). Найди решение той же задачи методом Монте-Карло. Сравни результаты и сделай выводы.

**1.3** (1,5 балла). Найди минимальное N, такое, что с вероятностью минимум 80% система не перегружена. Можно использовать аналитический метод или метод Монте-Карло.

*Указание:* при реализации п. 1.3 можно использовать функции из предыдущих пунктов и реализовывать свои методы.


In [2]:
### 1.1

In [3]:
from functools import cache
from math import comb


@cache
def count_stable_distributions(
        servers_left,
        requests_left,
        servers_with_3_or_more,
        servers_with_4_or_more,
):
    """
    А я зря на черные алгосы хожу чтоли? Сидел 8 часов решал задачи по рекурсии и вот скоро дедлайн по теорверу
    Ну и чего бы не применить, зато вон какое красивое решение получилось. По сути брутфорс перебор дерева рекурсией с мемоизацией
    """
    if servers_left == 0: return requests_left == 0
    return sum(
        comb(requests_left, x)
        * count_stable_distributions(
            servers_left - 1,
            requests_left - x,
            servers_with_3_or_more + (x >= 3),
            servers_with_4_or_more + (x >= 4),
        )
        for x in range(requests_left + 1)
        if servers_with_3_or_more + (x >= 3) < 3
        and servers_with_4_or_more + (x >= 4) < 2
    )

def calculate_stability_probability(
    N_servers: int,
    k_requests: int
) -> float:
    """Вычисляет вероятность стабильности системы
    """
    return count_stable_distributions(N_servers, k_requests, 0, 0) / N_servers ** k_requests

In [4]:
N_SERVERS = 30
K_RPS = 10

favorable_cases = count_stable_distributions(N_SERVERS, K_RPS, 0, 0)
stability_prob = calculate_stability_probability(N_SERVERS, K_RPS)

In [5]:
##### Вывод результатов. Ничего менять здесь не нужно
print(f"━━━ Подсчёт вероятности стабильности системы ━━━")
print(f"  {'Условие 1 :':<{w}} на три сервера упало 3 запроса и более")
print(f"  {'Условие 2 :':<{w}} на два сервера упало 4 запроса и более")
print(f"  {'Количество удовлетворительных комбинаций:':<{w}} {favorable_cases}")
print(f"  {'Аналитическая вероятность стабильности системы:':<{w}} {stability_prob}")

━━━ Подсчёт вероятности стабильности системы ━━━
  Условие 1 :                    на три сервера упало 3 запроса и более
  Условие 2 :                    на два сервера упало 4 запроса и более
  Количество удовлетворительных комбинаций: 590487001966080
  Аналитическая вероятность стабильности системы: 0.9999949228032312


In [6]:
### 1.2

In [7]:
from typing import Callable

def condition_lt_3_3(req_distribution: np.ndarray) -> bool:
    return np.sum(req_distribution >= 3) < 3

def condition_lt_4_2(req_distribution: np.ndarray) -> bool:
    return np.sum(req_distribution >= 4) < 2

def monte_carlo(
    N_servers: int,
    k_requests: int,
    conditions: list[Callable[[np.ndarray], bool]],
    iterations: int = 1_000_000,
) -> float:
    """Оценивает вероятность события методом Монте-Карло."""

    success = 0
    for _ in range(iterations):
        req_dist = np.random.multinomial(
            k_requests,
            [1 / N_servers] * N_servers # список с вероятностями где каждый элемент 1/N
        )
        if all(condition(req_dist) for condition in conditions):
            success += 1

    return success / iterations

In [8]:
total_servers, total_requests = 25, 10
stability_prob_mc = monte_carlo(total_servers, total_requests, [condition_lt_3_3, condition_lt_4_2])

In [9]:
##### Здесь находятся выводы. Менять ничего не нужно.
print(f"━━━ Симуляция Монте-Карло (серверы = {total_servers}, запросы = {total_requests}) ━━━")
print(f"  {'Вероятность Монте-Карло:':<{w}} {stability_prob_mc:.6f} (примерно {stability_prob_mc*100:.3f}%)")
print(f"  {'Аналитическая вероятность':<{w}} {stability_prob:.6f} (примерно {stability_prob*100:.3f}%)")
print(f"  {'Абсолютная разность':<{w}} {stability_prob - stability_prob_mc:.6f}")
print(f"  {'Относительная разность':<{w}} {(stability_prob - stability_prob_mc) * 100 / stability_prob:.4f}%")

━━━ Симуляция Монте-Карло (серверы = 25, запросы = 10) ━━━
  Вероятность Монте-Карло:       0.999988 (примерно 99.999%)
  Аналитическая вероятность      0.999995 (примерно 99.999%)
  Абсолютная разность            0.000007
  Относительная разность         0.0007%


**Выводы:**

1. Разница в выводах двух методах преребрежительно мала

2. Наша система на 25 серверах очень стабильна)

In [10]:
### 1.3

In [11]:
##### Твой код здесь
# ну у меня так получилось, что аналитический метод работает в 500 раз быстрее Монте-Карло, так что буду аналитическим проверять
# ДОРОГИЕ ЖЮРИ, ПРОЧИТАЙТЕ
# В условии сказано, что есть 2 условия нестабильности:
# - на 3 сервера больше 3х запросов
# - на 2 сервера больше 4х запросов
# А если у нас всего 1 сервер? Это блин тривиальный случай который является правильным ответом на пункт 1.3
# При 1 сервере система стабильна при любых RPS!!! Пропишите в условии что я не прав
print(f"Стабильность системы при 1 сервере и 10 RPS: {calculate_stability_probability(1, 100) * 100:.2f}%")
print(f"Стабильность системы при 2 серверах и 10 RPS: {calculate_stability_probability(2, 100) * 100:.2f}%")
# это ЯВНО не должно так работать, но вот такая вот задача
# чтоб удовлетворить такую задачу я начну перебор с 2х серверов, но это фул осознанный выбор

n = 2
v = calculate_stability_probability(n, 10)
while v < 0.8:
    n += 1
    v = calculate_stability_probability(n, 10)

print("...")
print(f"Стабильность системы при {n-1} серверах и 10 RPS: {calculate_stability_probability(n-1, 10) * 100:.2f}%")
print(f"Стабильность системы при {n} серверах и 10 RPS: {calculate_stability_probability(n, 10) * 100:.2f}%")


Стабильность системы при 1 сервере и 10 RPS: 100.00%
Стабильность системы при 2 серверах и 10 RPS: 0.00%
...
Стабильность системы при 4 серверах и 10 RPS: 78.31%
Стабильность системы при 5 серверах и 10 RPS: 91.52%


# Задача 2 вариант 1 (буква А)

## Задача 2 (2 балла). Геометрическая вероятность


Дан квадрат со стороной $a=5$, в который вписан круг. В квадрат случайно бросают точку.

**Задание:**

**2.1** (1 балл). Найди геометрическую вероятность того, что точка окажется внутри круга. Реши задачу аналитически.

**2.2** (1 балл). Реши ту же задачу методом Монте-Карло. Сравни результаты и сделай выводы.

In [12]:
a = 5.0

In [13]:
### 2.1

In [14]:
from math import pi

def geom_probability(
    a: float
) -> float:
    """Вычисляет вероятность выполнения условия для квадрата со стороной а
    курс школьной геометрии подсказывает что сторона квадрата нам не нужна
    оставлю код для нахождения через нее тут же, но оно реально не роляет ибо сокращается
    """
    # square_area = a ** 2
    # circle_area = pi*((a / 2) ** 2)
    # prob = square_area / circle_area
    return pi / 4


In [15]:
prob = geom_probability(a)

In [16]:
##### Вывод результатов. Ничего менять здесь не нужно
print(f"━━━ Подсчёт геометрической вероятности ━━━")
print(f"  {'Сторона квадрата:':<{w}} {a}")
print(f"  {'Геометрическая вероятность попадания точки в круг:':<{w}} {prob}")

━━━ Подсчёт геометрической вероятности ━━━
  Сторона квадрата:              5.0
  Геометрическая вероятность попадания точки в круг: 0.7853981633974483


In [17]:
### 2.2

In [18]:
def monte_carlo_geom(
    a: float,
    iterations: int = 1_000_000,
) -> float:
    """Оценивает вероятность события методом Монте-Карло."""
    rng = np.random.default_rng()
    dots = rng.uniform(-a / 2, a / 2, size=(iterations, 2))
    dist = np.sum(dots**2, axis=1)
    return np.sum(dist <= (a/2)**2) / iterations


In [19]:
prob_mc = monte_carlo_geom(a)

In [20]:
##### Здесь находятся выводы. Менять ничего не нужно.
print(f"━━━ Симуляция Монте-Карло (квадрат со стороной а = {a}) ━━━")
print(f"  {'Вероятность Монте-Карло:':<{w}} {prob_mc:.6f} (примерно {prob_mc*100:.3f}%)")
print(f"  {'Аналитическая вероятность':<{w}} {prob:.6f} (примерно {prob*100:.3f}%)")
print(f"  {'Абсолютная разность':<{w}} {prob - prob_mc:.6f}")
print(f"  {'Относительная разность':<{w}} {(prob - prob_mc)  * 100 / prob:.4f}%")

━━━ Симуляция Монте-Карло (квадрат со стороной а = 5.0) ━━━
  Вероятность Монте-Карло:       0.785059 (примерно 78.506%)
  Аналитическая вероятность      0.785398 (примерно 78.540%)
  Абсолютная разность            0.000339
  Относительная разность         0.0432%


**Выводы:**

1. Аналитически задача решается за константное время (формально за бесконечное.. нам к счастью не надо считать все цифры иррационального непериодического числа пи)

2. Методом монте карло мы посчитали с достаточно хорошей точностью